In [60]:
import pandas as pd
import numpy as np
from pathlib import Path
from trading_strategy.utils.metrics import get_metrics_for_clf
from collections import defaultdict
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv(Path('__file__').resolve().parents[3] / "resources" / "data.csv")

- roll_mean_10d: Close price rolling mean for 10 days
- roll_mean_20d: Close price rolling mean for 20 days
- growing_moving_average: if mean price for 10 days more than mean price for 20 days (1 if roll_mean_10d > roll_mean_20d 1, else 0)
- high_minus_low_relative: (High price for day - Low price) / Close price
- growth_{days}d, days=[1,3,7,30,90,252,365]: how much growth price since X day to today (Close price / Close price for previous X day)
- volatility: std for 30 days for (Close price / Close price for previous day) * sqrt 252 trading days in a year (growth_1d.rolling(30).std() * np.sqrt(252))
- sharpe: (growth_252d - RISK_FREE_RATE=0.045) / volatility
- RSI: Relative Strength Index (smoothed measure of recent gains vs. recent losses over the specified period)
- ADX: Average Directional Index, measures trend strength (0–20 → Weak/No trend; 20–40 → Developing/Moderate trend; 40–60 → Strong trend; 60+ → Very strong trend.)
- +DI: Positive Directional Indicator (high → strong upward trend; low → weak upward trend (or possibly downward trend))
- -DI: Negative Directional Movement (high → strong downward trend; low → weak downward trend (or possibly upward trend))
- MACD: Moving Average Convergence Divergence
- MACD_signal: MACD line crossing above Signal line → Bullish signal (momentum up); MACD line crossing below Signal line → Bearish signal (momentum down).
- MACD_hist: momentum strength (positive = bullish, negative = bearish)
- CCI: Commodity Channel Index (CCI > +100 → Strong bullish signal (overbought / upward momentum); CCI < -100 → Strong bearish signal (oversold / downward momentum); Crossing ±100 → Often used as entry/exit signals.)
- CMO: Chande Momentum Oscillator ([-100, +100]: CMO > +50 → Strong bullish momentum; CMO < -50 → Strong bearish momentum; CMO crossing above 0 → Possible start of an uptrend; CMO crossing below 0 → Possible start of a downtrend.)
- TRIX: Triple-smoothed exponential moving average of the price (Positive → uptrend gaining strength; Negative → downtrend gaining strength)
- ULTOSC: Ultimate Oscillator, capture momentum across three different timeframes to reduce false signals (overbought (>70) and oversold (<30) conditions)
- OBV: On-Balance Volume, relates price movement to volume (A rising OBV: volume is heavier on up days → potential buying pressure; A falling OBV: volume is heavier on down days → potential selling pressure.)
- AD: Accumulation/Distribution (A/D) Line, detect whether a stock is being accumulated or distributed, even if the price itself isn’t moving much (Rising A/D → accumulation (buying pressure), Falling A/D → distribution (selling pressure).)
- ADOSC: Accumulation/Distribution Oscillator, measures the momentum of accumulation/distribution using volume and price (Positive values → buying pressure (accumulation increasing); Negative values → selling pressure (distribution increasing); Crossovers of zero or the signal line: potential trend changes)
- ATR: Average True Range, how much the price typically moves in a given period, accounting for gaps and intraday swings (Higher ATR → higher price volatility; Lower ATR → lower price volatility)
- NATR: Normalized Average True Range, how volatile an asset is relative to its price (typical price movement over the period is about X% of the closing price.)
- AvgPrice: Average price of a candlestick (or bar) for each period
- MedPrice: Median price of a candlestick (or bar) for each period
- TypPrice: Typical price for a given period
- WCLPrice: Weighted close price for a given period
- Doji: Doji candlestick patterns, where the open and close prices are almost equal.
- Engulfing: Engulfing candlestick patterns (1 → Bullish Engulfing, -1 → Bearish Engulfing, 0 → No Engulfing pattern)
- Hammer: Hammer candlestick patterns (1 if hammer pattern detected: (Lower shadow at least twice the body → long lower wick. Body is ≤ 30% of total candle range → small real body.) 0 otherwise)
- ShootingStar: Shooting Star candlestick patterns (1 if a Shooting Star is detected: (Upper shadow at least twice the body → long upper wick. Body ≤ 30% of total candle range → small real body.) 0 otherwise.)
- Momentum: Measure of price change speed or trend (difference between the current value and the value period steps ago)
- ROC: Rate of Change
- ROCP: Rate of Change in proportion
- ROCR: Rate of Change Ratio, relative change expressed as a ratio (1 → no change over the period; Greater than 1 → the series increased over the period; Less than 1 → the series decreased over the period.)
- ROCR100: Rate of Change Ratio expressed as a percentage
- WilliamsR: Williams Percent Range, measures overbought and oversold levels ([0, -100]: Above -20 → asset may be overbought; Below -80 → asset may be oversold; It compares the current close to the recent high-low range, giving a sense of momentum and reversal potential.)
- ADXR: Average Directional Movement Rating, measures trend strength with a bit of "lag" to reduce noise. often used to filter out false signals in choppy markets.
- APO: Absolute Price Oscillator (APO > 0 → Upward momentum (short EMA above long EMA). APO < 0 → Downward momentum (short EMA below long EMA). Crossing zero → Possible trend reversal signal.)
- PPO: Percentage Price Oscillator (PPO > 0 → Short-term EMA is above long-term EMA → bullish momentum; PPO < 0 → Short-term EMA is below long-term EMA → bearish momentum; Crossing zero line → Potential trend reversal.)
- StochK: Stochastic Oscillator ([0, 100]: %K or %D > 80 → Overbought (possible reversal down); %K or %D < 20 → Oversold (possible reversal up)
- StochD: %K crossing above %D → Bullish signal; %K crossing below %D → Bearish signal.
- StochFastK: Stochastic Oscillator 
- StochFastD: Stochastic Oscillator 
- StochRSI_K: Stochastic RSI ([0, 1]: > 0.8 → Overbought, < 0.2 → Oversold.)
- StochRSI_D: Stochastic RSI ([0, 1]: > 0.8 → Overbought, < 0.2 → Oversold.)
- gdppot_us_yoy: Real Potential Gross Domestic Product yearly growth
- gdppot_us_qoq: Real Potential Gross Domestic Product quaterly growth
- cpi_core_yoy: Core CPI index yearly growth
- cpi_core_mom: Core CPI index montly growth
- growth_FEDFUNDS_Xd, X=[1,3,7,30,90,252,365]: Fed rate growth
- growth_DGS1_Xd: DGS1 growth
- growth_DGS5_Xd: DGS5 growth
- growth_DGS10_Xd: DGS10 growth
- spx_dod: S&P500 daily growth
- spx_qoq: S&P500 quaterly growth
- spx_yoy: S&P500 yearly growth
- growth_^GDAXI_Xd: 40 largest German companies growth
- growth_^GSPC_Xd: SNP Real Time Price
- growth_^DJI_Xd: Dow Jones Industrial Average - 30 large, publicly owned blue-chip companies trading on the New York Stock Exchange (NYSE) and Nasdaq
- growth_VOO_Xd: VOO
- growth_EPI_Xd: WisdomTree India Earnings Fund
- growth_^VIX_Xd: Volatility Index
- growth_GC=F_Xd: GOLD
- growth_CL=F_Xd: WTI Crude Oil
- growth_BZ=F_Xd: Brent Oil
- growth_BTC-USD_Xd: BTC-USD

In [3]:
df['Date'].min(), df['Date'].max()

('2010-01-04', '2025-07-31')

In [4]:
columns = ['roll_mean_10d', 'roll_mean_20d','growing_moving_average', 'high_minus_low_relative', 
           'growth_1d','growth_3d','growth_7d','growth_30d','growth_90d','growth_252d','growth_365d','volatility','sharpe',
           'RSI','ADX','+DI','-DI','MACD','MACD_signal','MACD_hist','CCI','CMO','TRIX','ULTOSC','OBV','AD','ADOSC','ATR','NATR',
           'AvgPrice','MedPrice','TypPrice','WCLPrice','Doji','Engulfing','Hammer','ShootingStar','Momentum','ROC','ROCP','ROCR','ROCR100',
           'WilliamsR','ADXR','APO','PPO','StochK','StochD','StochFastK','StochFastD','StochRSI_K','StochRSI_D','year','month','weekday',
           'gdppot_us_yoy','gdppot_us_qoq','cpi_core_yoy','cpi_core_mom','fedfunds_yoy','fedfunds_mom',
           'growth_DGS1_1d','growth_DGS1_3d','growth_DGS1_7d','growth_DGS1_30d','growth_DGS1_90d','growth_DGS1_252d','growth_DGS1_365d',
           'growth_DGS5_1d','growth_DGS5_3d','growth_DGS5_7d','growth_DGS5_30d','growth_DGS5_90d','growth_DGS5_252d','growth_DGS5_365d',
           'growth_DGS10_1d','growth_DGS10_3d','growth_DGS10_7d','growth_DGS10_30d','growth_DGS10_90d','growth_DGS10_252d','growth_DGS10_365d',
           'spx_dod','spx_qoq','spx_yoy',
           'growth_^GDAXI_1d','growth_^GDAXI_7d','growth_^GDAXI_30d','growth_^GDAXI_90d','growth_^GDAXI_252d','growth_^GDAXI_365d',
           'growth_^GSPC_1d','growth_^GSPC_7d','growth_^GSPC_30d','growth_^GSPC_90d','growth_^GSPC_252d','growth_^GSPC_365d',        
           'growth_^DJI_1d','growth_^DJI_3d','growth_^DJI_7d','growth_^DJI_30d','growth_^DJI_90d','growth_^DJI_252d','growth_^DJI_365d',
           'growth_VOO_1d','growth_VOO_3d','growth_VOO_7d','growth_VOO_30d','growth_VOO_90d','growth_VOO_252d','growth_VOO_365d',
           'growth_EPI_1d','growth_EPI_3d','growth_EPI_7d','growth_EPI_30d','growth_EPI_90d','growth_EPI_252d','growth_EPI_365d',
           'growth_^VIX_1d','growth_^VIX_3d','growth_^VIX_7d','growth_^VIX_30d','growth_^VIX_90d','growth_^VIX_252d','growth_^VIX_365d',
           'growth_GC=F_1d','growth_GC=F_3d','growth_GC=F_7d','growth_GC=F_30d','growth_GC=F_90d','growth_GC=F_252d','growth_GC=F_365d',
           'growth_CL=F_1d','growth_CL=F_3d','growth_CL=F_7d','growth_CL=F_30d','growth_CL=F_90d','growth_CL=F_252d','growth_CL=F_365d',
           'growth_BZ=F_1d','growth_BZ=F_3d','growth_BZ=F_7d','growth_BZ=F_30d','growth_BZ=F_90d','growth_BZ=F_252d','growth_BZ=F_365d',
           'growth_BTC-USD_1d','growth_BTC-USD_3d','growth_BTC-USD_7d','growth_BTC-USD_30d','growth_BTC-USD_90d','growth_BTC-USD_252d','growth_BTC-USD_365d']

numeric = ['roll_mean_10d', 'roll_mean_20d','high_minus_low_relative',
           'growth_1d','growth_3d','growth_7d','growth_30d','growth_90d','growth_252d','growth_365d','volatility','sharpe',
           'RSI','ADX','+DI','-DI','MACD','MACD_signal','MACD_hist','CCI','CMO','TRIX','ULTOSC','OBV','AD','ADOSC','ATR','NATR',
           'AvgPrice','MedPrice','TypPrice','WCLPrice','Momentum','ROC','ROCP','ROCR','ROCR100',
           'WilliamsR','ADXR','APO','PPO','StochK','StochD','StochFastK','StochFastD','StochRSI_K','StochRSI_D',
           'gdppot_us_yoy','gdppot_us_qoq','cpi_core_yoy','cpi_core_mom','fedfunds_yoy','fedfunds_mom',
           'growth_DGS1_1d','growth_DGS1_3d','growth_DGS1_7d','growth_DGS1_30d','growth_DGS1_90d','growth_DGS1_252d','growth_DGS1_365d',
           'growth_DGS5_1d','growth_DGS5_3d','growth_DGS5_7d','growth_DGS5_30d','growth_DGS5_90d','growth_DGS5_252d','growth_DGS5_365d',
           'growth_DGS10_1d','growth_DGS10_3d','growth_DGS10_7d','growth_DGS10_30d','growth_DGS10_90d','growth_DGS10_252d','growth_DGS10_365d',
           'spx_dod','spx_qoq','spx_yoy',
           'growth_^GDAXI_1d','growth_^GDAXI_7d','growth_^GDAXI_30d','growth_^GDAXI_90d','growth_^GDAXI_252d','growth_^GDAXI_365d',
           'growth_^GSPC_1d','growth_^GSPC_7d','growth_^GSPC_30d','growth_^GSPC_90d','growth_^GSPC_252d','growth_^GSPC_365d',
           'growth_^DJI_1d','growth_^DJI_3d','growth_^DJI_7d','growth_^DJI_30d','growth_^DJI_90d','growth_^DJI_252d','growth_^DJI_365d',
           'growth_VOO_1d','growth_VOO_3d','growth_VOO_7d','growth_VOO_30d','growth_VOO_90d','growth_VOO_252d','growth_VOO_365d',
           'growth_EPI_1d','growth_EPI_3d','growth_EPI_7d','growth_EPI_30d','growth_EPI_90d','growth_EPI_252d','growth_EPI_365d',
           'growth_^VIX_1d','growth_^VIX_3d','growth_^VIX_7d','growth_^VIX_30d','growth_^VIX_90d','growth_^VIX_252d','growth_^VIX_365d',
           'growth_GC=F_1d','growth_GC=F_3d','growth_GC=F_7d','growth_GC=F_30d','growth_GC=F_90d','growth_GC=F_252d','growth_GC=F_365d',
           'growth_CL=F_1d','growth_CL=F_3d','growth_CL=F_7d','growth_CL=F_30d','growth_CL=F_90d','growth_CL=F_252d','growth_CL=F_365d',
           'growth_BZ=F_1d','growth_BZ=F_3d','growth_BZ=F_7d','growth_BZ=F_30d','growth_BZ=F_90d','growth_BZ=F_252d','growth_BZ=F_365d',
           'growth_BTC-USD_1d','growth_BTC-USD_3d','growth_BTC-USD_7d','growth_BTC-USD_30d','growth_BTC-USD_90d','growth_BTC-USD_252d','growth_BTC-USD_365d']

binary = ['growing_moving_average','Doji','Hammer','ShootingStar']
categorical = ['Engulfing','year','month','weekday',]
idx = ['Date','ticker']

to_predict_numerical = ['growth_future_1d','growth_future_3d','growth_future_7d','growth_future_30d','growth_future_90d','growth_future_252d','growth_future_365d']
to_predict_binary = ['is_positive_growth_1d_future','is_positive_growth_3d_future','is_positive_growth_7d_future','is_positive_growth_30d_future',
                     'is_positive_growth_90d_future','is_positive_growth_252d_future','is_positive_growth_365d_future',]

In [5]:
assert len(numeric) + len(binary) + len(categorical) == len(columns)

In [6]:
print(f'Total features size: {len(columns)}')
print(f'Total predict size: {len(to_predict_numerical)}')

Total features size: 153
Total predict size: 7


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1101534 entries, 0 to 1101533
Columns: 180 entries, Date to growth_BTC-USD_365d
dtypes: float64(163), int64(15), object(2)
memory usage: 1.5+ GB


In [8]:
nans = df[columns].isna().sum()
nans[nans != 0].sort_values().tail(20)

growth_CL=F_365d        97034
growth_GC=F_365d        97315
growth_BZ=F_365d       105648
growth_365d            109080
growth_VOO_252d        112231
growth_^GDAXI_365d     113197
growth_DGS10_252d      113660
growth_DGS1_252d       113660
growth_DGS5_252d       113660
growth_DGS10_365d      141338
growth_DGS1_365d       141338
growth_DGS5_365d       141338
growth_VOO_365d        142410
growth_BTC-USD_1d      317829
growth_BTC-USD_3d      318373
growth_BTC-USD_7d      318919
growth_BTC-USD_30d     323560
growth_BTC-USD_90d     334753
growth_BTC-USD_252d    364822
growth_BTC-USD_365d    386468
dtype: int64

In [9]:
infs1 = (df == np.inf).sum()
infs1[infs1 != 0]

sharpe         7
WilliamsR     11
StochK        11
StochFastK    53
dtype: int64

In [10]:
infs2 = (df == -np.inf).sum()
infs2[infs2 != 0]

StochFastK    11
dtype: int64

In [11]:
df = df.replace([np.inf, -np.inf], np.nan)

### EDA

##### Correlations
drop highly corr features with each other

In [12]:
columns = ['growing_moving_average', 'high_minus_low_relative', 
           'growth_1d','growth_3d','growth_7d','growth_30d','growth_90d','growth_252d','growth_365d','volatility','sharpe',
           'RSI','ADX','+DI','-DI','MACD_signal','MACD_hist','CCI','TRIX','ULTOSC','OBV','ADOSC','ATR','NATR',
           'AvgPrice','Doji','Engulfing','Hammer','ShootingStar','Momentum','ROC',
           'ADXR','StochFastK','StochFastD','StochRSI_K','year','month','weekday',
           'gdppot_us_qoq','cpi_core_yoy','cpi_core_mom','fedfunds_yoy','fedfunds_mom',
           'growth_DGS1_1d','growth_DGS1_3d','growth_DGS1_7d','growth_DGS1_30d','growth_DGS1_90d','growth_DGS1_252d',
           'growth_DGS5_1d','growth_DGS5_3d','growth_DGS5_7d','growth_DGS5_30d','growth_DGS5_90d','growth_DGS5_252d','growth_DGS5_365d','spx_qoq',
           'growth_^GDAXI_1d','growth_^GDAXI_7d','growth_^GDAXI_30d','growth_^GDAXI_90d','growth_^GDAXI_252d','growth_^GDAXI_365d',
           'growth_^GSPC_1d','growth_^GSPC_7d','growth_^GSPC_30d','growth_^GSPC_90d','growth_^GSPC_252d','growth_^GSPC_365d',        
           'growth_EPI_1d','growth_EPI_3d','growth_EPI_7d','growth_EPI_30d','growth_EPI_90d','growth_EPI_252d','growth_EPI_365d',
           'growth_^VIX_1d','growth_^VIX_3d','growth_^VIX_7d','growth_^VIX_30d','growth_^VIX_90d','growth_^VIX_252d','growth_^VIX_365d',
           'growth_GC=F_1d','growth_GC=F_3d','growth_GC=F_7d','growth_GC=F_30d','growth_GC=F_90d','growth_GC=F_252d','growth_GC=F_365d',
           'growth_CL=F_1d','growth_CL=F_3d','growth_CL=F_7d','growth_CL=F_30d','growth_CL=F_90d','growth_CL=F_252d','growth_CL=F_365d',
           'growth_BZ=F_1d','growth_BZ=F_3d','growth_BZ=F_7d','growth_BZ=F_30d','growth_BZ=F_90d','growth_BZ=F_252d','growth_BZ=F_365d',
           'growth_BTC-USD_1d','growth_BTC-USD_3d','growth_BTC-USD_7d','growth_BTC-USD_30d','growth_BTC-USD_90d','growth_BTC-USD_252d','growth_BTC-USD_365d',
          # 'roll_mean_20d', 'StochD', 'StochK', 'WilliamsR','CMO','MACD','roll_mean_10d', 'MedPrice','TypPrice','WCLPrice','APO','AD','ROCP','ROCR',
          # 'ROCR100','PPO','StochRSI_D','gdppot_us_yoy','growth_DGS1_365d','growth_DGS10_252d','growth_DGS10_1d','growth_DGS10_3d','growth_DGS10_7d',
          # 'growth_VOO_1d','growth_VOO_3d','growth_VOO_7d','growth_VOO_30d','growth_VOO_90d','growth_VOO_252d','growth_VOO_365d',
          # 'growth_DGS10_30d','growth_DGS10_90d','growth_DGS10_365d','spx_dod','spx_yoy',
          # 'growth_^DJI_1d','growth_^DJI_3d','growth_^DJI_7d','growth_^DJI_30d','growth_^DJI_90d','growth_^DJI_252d','growth_^DJI_365d',
          ]

corr_feats = df[columns + to_predict_numerical + to_predict_binary].corr()

In [16]:
feat = 'gdppot_us_qoq'
corr_feats[abs(corr_feats[feat]) > 0.8][feat]

year             0.921668
gdppot_us_qoq    1.000000
Name: gdppot_us_qoq, dtype: float64

In [17]:
corr_feats[abs(corr_feats['growth_future_30d']) > 0.1]['growth_future_30d']

high_minus_low_relative           0.114904
volatility                        0.145502
NATR                              0.162004
fedfunds_mom                     -0.114807
spx_qoq                          -0.112097
growth_^GSPC_90d                 -0.103833
growth_^VIX_90d                   0.131406
growth_^VIX_252d                  0.145263
growth_^VIX_365d                  0.124459
growth_future_1d                  0.183173
growth_future_3d                  0.315753
growth_future_7d                  0.477677
growth_future_30d                 1.000000
growth_future_90d                 0.570099
growth_future_252d                0.356038
growth_future_365d                0.294143
is_positive_growth_1d_future      0.111688
is_positive_growth_3d_future      0.201519
is_positive_growth_7d_future      0.314131
is_positive_growth_30d_future     0.691701
is_positive_growth_90d_future     0.371936
is_positive_growth_252d_future    0.204618
is_positive_growth_365d_future    0.160993
Name: growt

In [18]:
corr_feats[abs(corr_feats['is_positive_growth_30d_future']) > 0.08]['is_positive_growth_30d_future']

cpi_core_yoy                     -0.086324
cpi_core_mom                     -0.082464
fedfunds_mom                     -0.085806
spx_qoq                          -0.080971
growth_future_1d                  0.124110
growth_future_3d                  0.213072
growth_future_7d                  0.326487
growth_future_30d                 0.691701
growth_future_90d                 0.375985
growth_future_252d                0.210020
growth_future_365d                0.161074
is_positive_growth_1d_future      0.099369
is_positive_growth_3d_future      0.183904
is_positive_growth_7d_future      0.295366
is_positive_growth_30d_future     1.000000
is_positive_growth_90d_future     0.363193
is_positive_growth_252d_future    0.206042
is_positive_growth_365d_future    0.172436
Name: is_positive_growth_30d_future, dtype: float64

##### dummies

In [19]:
categorical

['Engulfing', 'year', 'month', 'weekday']

In [20]:
df['bullish_engulfing'] = np.where(df['Engulfing'] == -1, 1, 0)
df['bearish_engulfing'] = np.where(df['Engulfing'] == 1, 1, 0)

In [21]:
for feat in ['year', 'month', 'weekday']:
    df = df.join(pd.get_dummies(df[feat], prefix=feat).astype(int))
    df.drop(feat, axis=1, inplace=True)

In [22]:
df.head()

,Date,Close,High,Low,Open,Volume,prev_close_price,ticker,growth_1d,growth_3d,...,month_8,month_9,month_10,month_11,month_12,weekday_0,weekday_1,weekday_2,weekday_3,weekday_4
0,2010-01-04,30.109993,30.446071,30.041404,30.411778,6109400.0,NaN,MDT,NaN,NaN,...,0,0,0,0,0,1,0,0,0,0
1,2010-01-05,30.727285,30.734146,30.007115,30.178584,6568600.0,30.109993,MDT,1.020501,NaN,...,0,0,0,0,0,0,1,0,0,0
2,2010-01-06,31.233728,31.323300,30.606709,30.792747,7470700.0,30.727285,MDT,1.016482,NaN,...,0,0,0,0,0,0,0,1,0,0
3,2010-01-07,31.523119,31.564462,31.020127,31.247506,6895200.0,31.233728,MDT,1.009265,1.046932,...,0,0,0,0,0,0,0,0,1,0
4,2010-01-08,31.688486,31.757388,31.316409,31.447324,4947200.0,31.523119,MDT,1.005246,1.031282,...,0,0,0,0,0,0,0,0,0,1


##### final dataset

In [23]:
binary = ['growing_moving_average','Doji','Hammer','ShootingStar','bullish_engulfing','bearish_engulfing',
          'year_2010','year_2011','year_2012','year_2013','year_2014','year_2015','year_2016','year_2017','year_2018',
          'year_2019','year_2020','year_2021','year_2022','year_2023','year_2024','year_2025',
          'month_1','month_2','month_3','month_4','month_5','month_6','month_7','month_8','month_9','month_10','month_11','month_12',
          'weekday_0','weekday_1','weekday_2','weekday_3','weekday_4']

numeric = ['high_minus_low_relative', 
           'growth_1d','growth_3d','growth_7d','growth_30d','growth_90d','growth_252d','growth_365d','volatility','sharpe',
           'RSI','ADX','+DI','-DI','MACD_signal','MACD_hist','CCI','TRIX','ULTOSC','OBV','ADOSC','ATR','NATR',
           'AvgPrice','Momentum','ROC',
           'ADXR','StochFastK','StochFastD','StochRSI_K',
           'gdppot_us_qoq','cpi_core_yoy','cpi_core_mom','fedfunds_yoy','fedfunds_mom',
           'growth_DGS1_1d','growth_DGS1_3d','growth_DGS1_7d','growth_DGS1_30d','growth_DGS1_90d','growth_DGS1_252d',
           'growth_DGS5_1d','growth_DGS5_3d','growth_DGS5_7d','growth_DGS5_30d','growth_DGS5_90d','growth_DGS5_252d','growth_DGS5_365d','spx_qoq',
           'growth_^GDAXI_1d','growth_^GDAXI_7d','growth_^GDAXI_30d','growth_^GDAXI_90d','growth_^GDAXI_252d','growth_^GDAXI_365d',
           'growth_^GSPC_1d','growth_^GSPC_7d','growth_^GSPC_30d','growth_^GSPC_90d','growth_^GSPC_252d','growth_^GSPC_365d',        
           'growth_EPI_1d','growth_EPI_3d','growth_EPI_7d','growth_EPI_30d','growth_EPI_90d','growth_EPI_252d','growth_EPI_365d',
           'growth_^VIX_1d','growth_^VIX_3d','growth_^VIX_7d','growth_^VIX_30d','growth_^VIX_90d','growth_^VIX_252d','growth_^VIX_365d',
           'growth_GC=F_1d','growth_GC=F_3d','growth_GC=F_7d','growth_GC=F_30d','growth_GC=F_90d','growth_GC=F_252d','growth_GC=F_365d',
           'growth_CL=F_1d','growth_CL=F_3d','growth_CL=F_7d','growth_CL=F_30d','growth_CL=F_90d','growth_CL=F_252d','growth_CL=F_365d',
           'growth_BZ=F_1d','growth_BZ=F_3d','growth_BZ=F_7d','growth_BZ=F_30d','growth_BZ=F_90d','growth_BZ=F_252d','growth_BZ=F_365d',
           'growth_BTC-USD_1d','growth_BTC-USD_3d','growth_BTC-USD_7d','growth_BTC-USD_30d','growth_BTC-USD_90d','growth_BTC-USD_252d','growth_BTC-USD_365d',
          ]

idx = ['Date','ticker']

to_predict_numerical = ['growth_future_30d']
to_predict_binary = ['is_positive_growth_30d_future']

In [24]:
df[idx + numeric + binary + to_predict_numerical + to_predict_binary].head()

,Date,ticker,high_minus_low_relative,growth_1d,growth_3d,growth_7d,growth_30d,growth_90d,growth_252d,growth_365d,...,month_10,month_11,month_12,weekday_0,weekday_1,weekday_2,weekday_3,weekday_4,growth_future_30d,is_positive_growth_30d_future
0,2010-01-04,MDT,0.013440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,1,0,0,0,0,0.994757,0
1,2010-01-05,MDT,0.023661,1.020501,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,1,0,0,0,0.982397,0
2,2010-01-06,MDT,0.022943,1.016482,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,1,0,0,0.964042,0
3,2010-01-07,MDT,0.017268,1.009265,1.046932,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,1,0,0.954318,0
4,2010-01-08,MDT,0.013916,1.005246,1.031282,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,1,0.932159,0


In [25]:
len(binary), len(numeric)

(39, 103)

### Model

In [26]:
df = df[~df['growth_future_30d'].isna()]

In [27]:
df.reset_index(drop=True, inplace=True)

##### Data split

In [28]:
# test data: 2024-01-01 - 2025-01-08
test_data = df[df['Date'] >= '2024-01-01']
not_test_data = df[df['Date'] < '2024-01-01']

In [29]:
train_1 = not_test_data[(not_test_data['Date'] >= '2010-01-01') & (not_test_data['Date'] < '2013-01-01')] 
val_1 = not_test_data[(not_test_data['Date'] >= '2013-01-01') & (not_test_data['Date'] < '2014-01-01')] 
train_2 = not_test_data[(not_test_data['Date'] >= '2010-01-01') & (not_test_data['Date'] < '2016-01-01')] 
val_2 = not_test_data[(not_test_data['Date'] >= '2016-01-01') & (not_test_data['Date'] < '2017-01-01')] 
train_3 = not_test_data[(not_test_data['Date'] >= '2010-01-01') & (not_test_data['Date'] < '2018-01-01')] 
val_3 = not_test_data[(not_test_data['Date'] >= '2018-01-01') & (not_test_data['Date'] < '2019-01-01')] 
train_4 = not_test_data[(not_test_data['Date'] >= '2010-01-01') & (not_test_data['Date'] < '2021-01-01')] 
val_4 = not_test_data[(not_test_data['Date'] >= '2021-01-01') & (not_test_data['Date'] < '2022-01-01')]
train_5 = not_test_data[(not_test_data['Date'] >= '2010-01-01') & (not_test_data['Date'] < '2023-01-01')] 
val_5 = not_test_data[(not_test_data['Date'] >= '2023-01-01') & (not_test_data['Date'] < '2024-01-01')]

In [30]:
feat = 'is_positive_growth_30d_future'

tables = []
for i in range(1, 6): 
    tc = globals()[f"train_{i}"][feat].value_counts()
    tc_norm = globals()[f"train_{i}"][feat].value_counts(normalize=True)
    tmp = pd.concat([tc, tc_norm], axis=1)
    tmp.columns = [f"train_{i}_count", f"train_{i}_freq"]
    tables.append(tmp)

result = pd.concat(tables, axis=1).fillna(0)
result

,train_1_count,train_1_freq,train_2_count,train_2_freq,train_3_count,train_3_freq,train_4_count,train_4_freq,train_5_count,train_5_freq
is_positive_growth_30d_future,,,,,,,,,,
1,123258,0.613468,248005,0.609689,340623,0.623315,470349,0.617471,551181,0.60618
0,77662,0.386532,158768,0.390311,205847,0.376685,291386,0.382529,358089,0.39382


In [31]:
tables = []
for i in range(1, 6): 
    vc = globals()[f"val_{i}"][feat].value_counts()
    vc_norm = globals()[f"val_{i}"][feat].value_counts(normalize=True)
    tmp = pd.concat([vc, vc_norm], axis=1)
    tmp.columns = [f"val_{i}_count", f"val_{i}_freq"]
    tables.append(tmp)

result = pd.concat(tables, axis=1).fillna(0)
result

,val_1_count,val_1_freq,val_2_count,val_2_freq,val_3_count,val_3_freq,val_4_count,val_4_freq,val_5_count,val_5_freq
is_positive_growth_30d_future,,,,,,,,,,
1,47482,0.695279,45629,0.656277,36401,0.514727,44817,0.607762,40346,0.545386
0,20810,0.304721,23898,0.343723,34318,0.485273,28924,0.392238,33631,0.454614


### Baseline
Buy when SMA10 is lower and intersects SMA20, sell when they interescts again in the upper side

In [32]:
pred = (train_1['growing_moving_average'] == 1).astype(int)
real = train_1['is_positive_growth_30d_future']
accuracy, roc_auc, precision, recall, cm = get_metrics_for_clf(real, pred, [0, 1])

In [33]:
print(f'accuracy: {round(accuracy, 3)}')
print(f'roc_auc: {round(roc_auc, 3)}')
print(f'precision: {round(precision, 3)}')
print(f'recall: {round(recall, 3)}')

accuracy: 0.521
roc_auc: 0.506
precision: 0.619
recall: 0.571


In [34]:
pd.DataFrame(cm)

,0,1
0,34315,43347
1,52929,70329


In [35]:
train_metrics = defaultdict(list)
val_metrics = defaultdict(list)
for idx, (train, val) in enumerate([[train_1, val_1], [train_2, val_2], [train_3, val_3], [train_4, val_4], [train_5, val_5]]):
    print(idx+1)
    pred = (train['growing_moving_average'] == 1).astype(int)
    real = train['is_positive_growth_30d_future']
    accuracy, roc_auc, precision, recall, _ = get_metrics_for_clf(real, pred)
    train_metrics['accuracy'].append(accuracy)
    train_metrics['roc_auc'].append(roc_auc)
    train_metrics['precision'].append(precision)
    train_metrics['recall'].append(recall)

    pred = (val['growing_moving_average'] == 1).astype(int)
    real = val['is_positive_growth_30d_future']
    accuracy, roc_auc, precision, recall, _ = get_metrics_for_clf(real, pred)
    val_metrics['accuracy'].append(accuracy)
    val_metrics['roc_auc'].append(roc_auc)
    val_metrics['precision'].append(precision)
    val_metrics['recall'].append(recall)

1
2
3
4
5


In [36]:
print('Train metrics:')
print(f'accuracy: {round(np.mean(train_metrics["accuracy"]), 3)}')
print(f'roc_auc: {round(np.mean(train_metrics["roc_auc"]), 3)}')
print(f'precision: {round(np.mean(train_metrics["precision"]), 3)}')
print(f'recall: {round(np.mean(train_metrics["recall"]), 3)}')
print()
print('Val metrics:')
print(f'accuracy: {round(np.mean(val_metrics["accuracy"]), 3)}')
print(f'roc_auc: {round(np.mean(val_metrics["roc_auc"]), 3)}')
print(f'precision: {round(np.mean(val_metrics["precision"]), 3)}')
print(f'recall: {round(np.mean(val_metrics["recall"]), 3)}')

Train metrics:
accuracy: 0.514
roc_auc: 0.495
precision: 0.61
recall: 0.577

Val metrics:
accuracy: 0.505
roc_auc: 0.482
precision: 0.589
recall: 0.568


In [37]:
pred = (test_data['growing_moving_average'] == 1).astype(int)
real = test_data['is_positive_growth_30d_future']
accuracy, roc_auc, precision, recall, cm = get_metrics_for_clf(real, pred, [0, 1])

print(f'accuracy: {round(accuracy, 3)}')
print(f'roc_auc: {round(roc_auc, 3)}')
print(f'precision: {round(precision, 3)}')
print(f'recall: {round(recall, 3)}')

accuracy: 0.504
roc_auc: 0.495
precision: 0.57
recall: 0.555


### Desicion tree

In [73]:
binary = [
    'growing_moving_average',
    'bullish_engulfing',
]

numeric = ['high_minus_low_relative', 
           'growth_1d','growth_3d','growth_7d','growth_30d','growth_90d','growth_252d','growth_365d',
           'RSI','+DI','-DI','MACD_signal','MACD_hist','CCI','TRIX','ULTOSC','OBV','ADOSC','ATR','NATR',
           'AvgPrice','Momentum','ROC','ADXR','StochFastK','StochFastD',
           'gdppot_us_qoq','cpi_core_yoy','fedfunds_mom',
           'growth_DGS1_3d','growth_DGS1_30d','growth_^GDAXI_7d','growth_^GDAXI_30d','growth_^GDAXI_90d',
           'growth_^GSPC_7d','growth_^GSPC_30d','growth_^GSPC_90d','growth_^GSPC_365d',        
           'growth_EPI_3d','growth_EPI_7d','growth_EPI_30d',
           'growth_^VIX_3d','growth_^VIX_7d','growth_^VIX_30d','growth_^VIX_90d',
           'growth_GC=F_1d','growth_GC=F_7d','growth_GC=F_30d','growth_GC=F_90d',
           'growth_CL=F_7d','growth_CL=F_30d',
           'growth_BZ=F_3d','growth_BZ=F_30d','growth_BZ=F_90d',
          ]

In [74]:
clf = DecisionTreeClassifier(random_state=0, max_depth=15)
clf.fit(train_1[numeric + binary], train_1['is_positive_growth_30d_future'])

pred = (clf.predict_proba(val_1[numeric + binary])[:, 1] > 0.5).astype(int)

In [75]:
real = val_1['is_positive_growth_30d_future']
accuracy, roc_auc, precision, recall, cm = get_metrics_for_clf(real, pred)

print(f'accuracy: {round(accuracy, 3)}')
print(f'roc_auc: {round(roc_auc, 3)}')
print(f'precision: {round(precision, 3)}')
print(f'recall: {round(recall, 3)}')

accuracy: 0.665
roc_auc: 0.535
precision: 0.713
recall: 0.867


In [76]:
feat_importance = pd.DataFrame({feat_name: [imp] for feat_name, imp in zip(clf.feature_names_in_, clf.feature_importances_)}).T\
    .sort_values(0, key=lambda x: abs(x), ascending=False)

In [77]:
feat_importance.head(10)

,0
fedfunds_mom,0.106745
gdppot_us_qoq,0.081213
cpi_core_yoy,0.070009
NATR,0.068490
growth_^GSPC_365d,0.059542
OBV,0.052314
AvgPrice,0.037421
growth_252d,0.033083
ATR,0.033027
growth_90d,0.031909


In [78]:
MAX_DEPTH = 10

train_metrics = defaultdict(list)
val_metrics = defaultdict(list)
for idx, (train, val) in enumerate([[train_1, val_1], [train_2, val_2], [train_3, val_3], [train_4, val_4], [train_5, val_5]]):
    print(idx+1)
    clf = DecisionTreeClassifier(random_state=0, max_depth=MAX_DEPTH)
    clf.fit(train[numeric + binary], train['is_positive_growth_30d_future'])
    pred = (clf.predict_proba(train[numeric + binary])[:, 1] > 0.5).astype(int)
    real = train['is_positive_growth_30d_future']
    accuracy, roc_auc, precision, recall, _ = get_metrics_for_clf(real, pred)
    train_metrics['accuracy'].append(accuracy)
    train_metrics['roc_auc'].append(roc_auc)
    train_metrics['precision'].append(precision)
    train_metrics['recall'].append(recall)

    pred = (clf.predict_proba(val[numeric + binary])[:, 1] > 0.5).astype(int)
    real = val['is_positive_growth_30d_future']
    accuracy, roc_auc, precision, recall, _ = get_metrics_for_clf(real, pred)
    val_metrics['accuracy'].append(accuracy)
    val_metrics['roc_auc'].append(roc_auc)
    val_metrics['precision'].append(precision)
    val_metrics['recall'].append(recall)

1
2
3
4
5


In [79]:
print('Train metrics:')
print(f'accuracy: {round(np.mean(train_metrics["accuracy"]), 3)}')
print(f'roc_auc: {round(np.mean(train_metrics["roc_auc"]), 3)}')
print(f'precision: {round(np.mean(train_metrics["precision"]), 3)}')
print(f'recall: {round(np.mean(train_metrics["recall"]), 3)}')
print()
print('Val metrics:')
print(f'accuracy: {round(np.mean(val_metrics["accuracy"]), 3)}')
print(f'roc_auc: {round(np.mean(val_metrics["roc_auc"]), 3)}')
print(f'precision: {round(np.mean(val_metrics["precision"]), 3)}')
print(f'recall: {round(np.mean(val_metrics["recall"]), 3)}')

Train metrics:
accuracy: 0.714
roc_auc: 0.657
precision: 0.71
recall: 0.906

Val metrics:
accuracy: 0.584
roc_auc: 0.513
precision: 0.613
recall: 0.839


In [80]:
clf = DecisionTreeClassifier(random_state=0, max_depth=MAX_DEPTH)
clf.fit(not_test_data[numeric + binary], not_test_data['is_positive_growth_30d_future'])

pred = (clf.predict_proba(test_data[numeric + binary])[:, 1] > 0.5).astype(int)
real = test_data['is_positive_growth_30d_future']
accuracy, roc_auc, precision, recall, cm = get_metrics_for_clf(real, pred, [0, 1])

print(f'accuracy: {round(accuracy, 3)}')
print(f'roc_auc: {round(roc_auc, 3)}')
print(f'precision: {round(precision, 3)}')
print(f'recall: {round(recall, 3)}')

accuracy: 0.527
roc_auc: 0.508
precision: 0.581
recall: 0.631


### Random Forest

In [138]:
binary = ['growing_moving_average','Doji','Hammer','ShootingStar','bullish_engulfing','bearish_engulfing',
          'year_2010','year_2011','year_2012','year_2013','year_2014','year_2015','year_2016','year_2017','year_2018',
          'year_2019','year_2020','year_2021','year_2022','year_2023','year_2024','year_2025',
          'month_1','month_2','month_3','month_4','month_5','month_6','month_7','month_8','month_9','month_10','month_11','month_12',
          'weekday_0','weekday_1','weekday_2','weekday_3','weekday_4']

numeric = ['volatility',
           'high_minus_low_relative', 
           'growth_1d','growth_3d','growth_7d','growth_30d','growth_90d','growth_252d','growth_365d',
           'sharpe',
           'RSI','ADX','+DI','-DI','MACD_signal','MACD_hist','CCI','TRIX','ULTOSC','OBV','ADOSC','ATR','NATR',
           'AvgPrice','Momentum','ROC',
           'ADXR','StochFastK','StochFastD','StochRSI_K',
           'gdppot_us_qoq','cpi_core_yoy','cpi_core_mom','fedfunds_yoy','fedfunds_mom',
           'growth_DGS1_1d','growth_DGS1_3d','growth_DGS1_7d','growth_DGS1_30d','growth_DGS1_90d','growth_DGS1_252d',
           'growth_DGS5_1d','growth_DGS5_3d','growth_DGS5_7d','growth_DGS5_30d','growth_DGS5_90d','growth_DGS5_252d','growth_DGS5_365d','spx_qoq',
           'growth_^GDAXI_1d','growth_^GDAXI_7d','growth_^GDAXI_30d','growth_^GDAXI_90d','growth_^GDAXI_252d','growth_^GDAXI_365d',
           'growth_^GSPC_1d','growth_^GSPC_7d','growth_^GSPC_30d','growth_^GSPC_90d',
           'growth_^GSPC_365d',        
           'growth_EPI_1d','growth_EPI_3d','growth_EPI_7d','growth_EPI_30d','growth_EPI_90d','growth_EPI_252d','growth_EPI_365d',
           'growth_^VIX_1d','growth_^VIX_3d','growth_^VIX_7d','growth_^VIX_30d','growth_^VIX_90d','growth_^VIX_252d','growth_^VIX_365d',
           'growth_GC=F_1d','growth_GC=F_3d','growth_GC=F_7d','growth_GC=F_30d','growth_GC=F_90d','growth_GC=F_252d','growth_GC=F_365d',
           'growth_CL=F_1d','growth_CL=F_3d','growth_CL=F_7d','growth_CL=F_30d','growth_CL=F_90d','growth_CL=F_252d','growth_CL=F_365d',
           'growth_BZ=F_1d','growth_BZ=F_3d','growth_BZ=F_7d','growth_BZ=F_30d','growth_BZ=F_90d','growth_BZ=F_252d',
           'growth_BTC-USD_1d','growth_BTC-USD_3d','growth_BTC-USD_7d','growth_BTC-USD_30d','growth_BTC-USD_90d',
           'growth_BTC-USD_365d',
          ]

In [139]:
clf = RandomForestClassifier(max_depth=20, random_state=0)
clf.fit(train_1[numeric + binary], train_1['is_positive_growth_30d_future'])

pred = (clf.predict_proba(val_1[numeric + binary])[:, 1] > 0.5).astype(int)

real = val_1['is_positive_growth_30d_future']
accuracy, roc_auc, precision, recall, cm = get_metrics_for_clf(real, pred)

print(f'accuracy: {round(accuracy, 3)}')
print(f'roc_auc: {round(roc_auc, 3)}')
print(f'precision: {round(precision, 3)}')
print(f'recall: {round(recall, 3)}')

accuracy: 0.494
roc_auc: 0.542
precision: 0.741
recall: 0.418


In [140]:
feat_importance = pd.DataFrame({feat_name: [imp] for feat_name, imp in zip(clf.feature_names_in_, clf.feature_importances_)}).T\
    .sort_values(0, key=lambda x: abs(x), ascending=False)

feat_importance

,0
NATR,0.029080
volatility,0.027793
OBV,0.024779
TRIX,0.024639
ATR,0.023850
...,...
year_2020,0.000000
year_2025,0.000000
year_2024,0.000000
year_2023,0.000000


In [141]:
MAX_DEPTH = 10

train_metrics = defaultdict(list)
val_metrics = defaultdict(list)
for idx, (train, val) in enumerate([[train_1, val_1], [train_2, val_2], [train_3, val_3], [train_4, val_4], [train_5, val_5]]):
    print(idx+1)
    clf = RandomForestClassifier(max_depth=MAX_DEPTH, random_state=0)
    clf.fit(train[numeric + binary], train['is_positive_growth_30d_future'])
    pred = (clf.predict_proba(train[numeric + binary])[:, 1] > 0.5).astype(int)
    real = train['is_positive_growth_30d_future']
    accuracy, roc_auc, precision, recall, _ = get_metrics_for_clf(real, pred)
    train_metrics['accuracy'].append(accuracy)
    train_metrics['roc_auc'].append(roc_auc)
    train_metrics['precision'].append(precision)
    train_metrics['recall'].append(recall)

    pred = (clf.predict_proba(val[numeric + binary])[:, 1] > 0.5).astype(int)
    real = val['is_positive_growth_30d_future']
    accuracy, roc_auc, precision, recall, _ = get_metrics_for_clf(real, pred)
    val_metrics['accuracy'].append(accuracy)
    val_metrics['roc_auc'].append(roc_auc)
    val_metrics['precision'].append(precision)
    val_metrics['recall'].append(recall)

1
2
3
4
5


In [142]:
print('Train metrics:')
print(f'accuracy: {round(np.mean(train_metrics["accuracy"]), 3)}')
print(f'roc_auc: {round(np.mean(train_metrics["roc_auc"]), 3)}')
print(f'precision: {round(np.mean(train_metrics["precision"]), 3)}')
print(f'recall: {round(np.mean(train_metrics["recall"]), 3)}')
print()
print('Val metrics:')
print(f'accuracy: {round(np.mean(val_metrics["accuracy"]), 3)}')
print(f'roc_auc: {round(np.mean(val_metrics["roc_auc"]), 3)}')
print(f'precision: {round(np.mean(val_metrics["precision"]), 3)}')
print(f'recall: {round(np.mean(val_metrics["recall"]), 3)}')

Train metrics:
accuracy: 0.721
roc_auc: 0.66
precision: 0.71
recall: 0.926

Val metrics:
accuracy: 0.512
roc_auc: 0.522
precision: 0.644
recall: 0.636


In [143]:
feat_importance = pd.DataFrame({feat_name: [imp] for feat_name, imp in zip(clf.feature_names_in_, clf.feature_importances_)}).T\
    .sort_values(0, key=lambda x: abs(x), ascending=False)

feat_importance.tail(20)

,0
growth_EPI_1d,0.000714
year_2014,0.000594
growth_CL=F_1d,0.000581
year_2012,0.000531
year_2021,0.000416
growing_moving_average,0.000409
year_2010,0.000086
Hammer,0.000075
ShootingStar,0.000075
Doji,0.000069


In [144]:
feat_importance

,0
gdppot_us_qoq,0.038281
fedfunds_mom,0.035938
growth_^GSPC_90d,0.027725
cpi_core_yoy,0.026934
growth_DGS1_252d,0.025948
...,...
bullish_engulfing,0.000031
weekday_0,0.000017
year_2025,0.000000
year_2024,0.000000


In [ ]:
Train metrics:
accuracy: 0.721
roc_auc: 0.66
precision: 0.71
recall: 0.926

Val metrics:
accuracy: 0.512
roc_auc: 0.522
precision: 0.644
recall: 0.636